# LLM GMC — Name That LLM baseline

This competition-linked notebook computes a reproducible local validation score. It does not submit predictions to the leaderboard.

Baseline: word + character TF-IDF features with class-balanced LinearSVC. If `prompt_id` is present, validation is grouped by prompt to reduce prompt leakage.

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
print('Dataset directory exists:', DATA_DIR.exists())
print('Dataset files:', sorted(p.name for p in DATA_DIR.glob('*')) if DATA_DIR.exists() else [])
assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found under {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
print('Train shape:', train.shape)
print('Columns:', train.columns.tolist())
assert {'text', 'model'}.issubset(train.columns), 'Expected text and model columns'
train['text'] = train['text'].fillna('').astype(str)
train['model'] = train['model'].astype(str)
print('Model classes:', train['model'].nunique())
display(train['model'].value_counts())

In [ ]:
if 'prompt_id' in train.columns and train['prompt_id'].notna().any():
    groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, valid_idx = next(splitter.split(train, groups=groups))
    split_method = 'GroupShuffleSplit by prompt_id'
else:
    train_idx, valid_idx = train_test_split(range(len(train)), test_size=0.20, random_state=42, stratify=train['model'])
    split_method = 'stratified random split (prompt_id unavailable)'

train_part = train.iloc[list(train_idx)].copy()
valid_part = train.iloc[list(valid_idx)].copy()
print('Validation method:', split_method)
print('Training rows:', len(train_part))
print('Validation rows:', len(valid_part))
print('Validation classes:', valid_part['model'].nunique())

In [ ]:
import re
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

# OOF confidence-gated Gemini specialist.
# The global model remains the default; the specialist is used only when
# the global prediction is a Gemini label and its top-1/top-2 margin is small.
texts = train['text'].fillna('').astype(str).to_numpy()
labels = train['model'].astype(str).to_numpy()
groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str).to_numpy()
gemini_labels = np.asarray(['gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash'])
thresholds = [0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 0.75, 1.00]


def style_one(text):
    words = re.findall(r"\b[\w']+\b", text.lower())
    n_words, n_chars = max(len(words),1), max(len(text),1)
    sentences = max(len(re.findall(r'[.!?]+', text)),1)
    lengths = np.asarray([len(w) for w in words], dtype=float)
    punct = '.,;:!?()[]{}"\'/-'
    return np.asarray([
        np.log1p(len(text)), np.log1p(len(words)), np.log1p(sentences),
        lengths.mean() if len(lengths) else 0., lengths.std() if len(lengths) else 0.,
        len(set(words))/n_words, sum(c.isdigit() for c in text)/n_chars,
        sum(c.isupper() for c in text)/n_chars, sum(c.isspace() for c in text)/n_chars,
        text.count('\\n')/n_chars, text.count('\\n\\n')/n_chars,
        *[text.count(c)/n_chars for c in punct], text.count('#')/n_chars,
        text.count('*')/n_chars, text.count('```')/n_chars,
    ], dtype=np.float32)

def make_features():
    return FeatureUnion([
        ('word', TfidfVectorizer(analyzer='word', ngram_range=(1,3), min_df=2, sublinear_tf=True, max_features=120000)),
        ('char', TfidfVectorizer(analyzer='char', ngram_range=(2,6), min_df=2, sublinear_tf=True, max_features=150000)),
    ])

style = np.vstack([style_one(x) for x in texts])
per_threshold = {str(t): [] for t in thresholds}
fold_details = []
for fold, (tr, va) in enumerate(GroupKFold(n_splits=5).split(texts, labels, groups=groups), 1):
    tf = make_features()
    Xtr_text = tf.fit_transform(texts[tr]); Xva_text = tf.transform(texts[va])
    sc = StandardScaler(); Xtr_style = sc.fit_transform(style[tr]); Xva_style = sc.transform(style[va])
    Xtr = hstack([Xtr_text, csr_matrix(Xtr_style)], format='csr')
    Xva = hstack([Xva_text, csr_matrix(Xva_style)], format='csr')
    ytr, yva = labels[tr], labels[va]

    global_model = LinearSVC(C=1.0, class_weight='balanced', max_iter=1200)
    global_model.fit(Xtr, ytr)
    pred_global = global_model.predict(Xva)
    decision = global_model.decision_function(Xva)
    ordered = np.sort(decision, axis=1)
    margins = ordered[:, -1] - ordered[:, -2]
    gemini_pred = np.isin(pred_global, gemini_labels)

    gm = np.isin(ytr, gemini_labels)
    specialist = LinearSVC(C=1.0, class_weight='balanced', max_iter=1200)
    specialist.fit(Xtr[gm], ytr[gm])
    pred_spec = specialist.predict(Xva)

    base_ba = balanced_accuracy_score(yva, pred_global)
    fold_row = {'fold': fold, 'global_balanced_accuracy': float(base_ba), 'gemini_predicted_rows': int(gemini_pred.sum()), 'thresholds': {}}
    for threshold in thresholds:
        route = gemini_pred & (margins <= threshold)
        pred_gate = pred_global.copy()
        pred_gate[route] = pred_spec[route]
        score = balanced_accuracy_score(yva, pred_gate)
        global_correct = pred_global == yva
        specialist_correct = pred_spec == yva
        fold_row['thresholds'][str(threshold)] = {
            'balanced_accuracy': float(score), 'routed_rows': int(route.sum()),
            'routed_percent': float(route.mean()*100),
            'global_correct_broken': int(np.sum(route & global_correct & ~specialist_correct)),
            'global_wrong_fixed': int(np.sum(route & ~global_correct & specialist_correct)),
        }
        per_threshold[str(threshold)].append(float(score))
    fold_details.append(fold_row)
    print(f"Fold {fold}: global={base_ba:.4f}, Gemini rows={gemini_pred.sum()}")
    for t in thresholds:
        d=fold_row['thresholds'][str(t)]
        print(f"  threshold={t:.2f}: gated={d['balanced_accuracy']:.4f}, routed={d['routed_percent']:.1f}%, fixed={d['global_wrong_fixed']}, broken={d['global_correct_broken']}")

summary = {
    'experiment': 'OOF confidence-gated Gemini specialist',
    'split_method': '5-fold GroupKFold by prompt_id',
    'thresholds': thresholds,
    'fold_details': fold_details,
    'global_mean': float(np.mean([x['global_balanced_accuracy'] for x in fold_details])),
    'global_std': float(np.std([x['global_balanced_accuracy'] for x in fold_details], ddof=1)),
    'threshold_summary': {
        str(t): {
            'mean_balanced_accuracy': float(np.mean(per_threshold[str(t)])),
            'std_balanced_accuracy': float(np.std(per_threshold[str(t)], ddof=1)),
            'fold_scores': per_threshold[str(t)],
        } for t in thresholds
    },
}
print('\\nSUMMARY:', json.dumps(summary, indent=2))
Path('/kaggle/working/gated_gemini_results.json').write_text(json.dumps(summary, indent=2))
print('Saved:', '/kaggle/working/gated_gemini_results.json')


## OOF confidence-gated Gemini specialist

This run sweeps fixed LinearSVC margin thresholds and applies the Gemini specialist only to low-margin global Gemini predictions.
